# Backbone Experiments
Run multiple backbone experiments with identical training/inference pipeline.


In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.inference as inference
import src.feature_cache as feature_cache
from src.models import ArcFaceHeadModel, build_backbone
from src.utils import get_device, set_seed
from src.training import fit_embeddings
from src.wandb_utils import init_wandb

# Load environment variables from .env file
load_dotenv(project_root / ".env")

RANDOM_SEED = 42
set_seed(RANDOM_SEED)

device = get_device()
device


device(type='cuda')

## Base Config


In [2]:
config = {
    # Paths
    "data_dir": Path("../data"),
    "checkpoint_dir": Path("../checkpoints"),

    # Model
    #"backbone_name": "hf-hub:BVRA/MegaDescriptor-L-384",
    #"input_size": 384,
    "embedding_dim": 256,
    "hidden_dim": 512,
    "freeze_backbone": True,

    # ArcFace
    "arcface_margin": 0.5,
    "arcface_scale": 64.0,
    "dropout": 0.3,

    # Training
    "batch_size": 32,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "num_epochs": 100,
    "patience": 10,
    "val_split": 0.2,
    "num_workers": 2,

    # Reproducibility
    "seed": RANDOM_SEED,

    # Normalization
    "mean": data.DEFAULT_MEAN,
    "std": data.DEFAULT_STD,
}

## Load Data


In [3]:
def load_data(config, model):
    train_df = data.load_train_df(config["data_dir"])
    train_df, label_encoder = data.encode_labels(train_df)
    num_classes = len(label_encoder.classes_)

    train_data, val_data = data.train_val_split(
        train_df,
        val_split=config["val_split"],
        seed=config["seed"],
        stratify_col="ground_truth",
    )

    backbone_train_loader, backbone_val_loader = data.create_backbone_dataloaders(
        model,
        train_data,
        val_data,
        img_dir=config["data_dir"] / "train" / "train",
        input_size=config["input_size"],
        batch_size=config["batch_size"],
        num_workers=config["num_workers"],
        mean=config["mean"],
        std=config["std"],
    )

    pairs_df = data.load_test_pairs_df(config["data_dir"])
    unique_images = set(pairs_df["query_image"].unique()) | set(pairs_df["gallery_image"].unique())
    unique_images = sorted(list(unique_images))

    test_df = pd.DataFrame({"filename": unique_images})

    test_loader = data.create_test_loader(
        model,
        test_df,
        img_dir=config["data_dir"] / "test" / "test",
        input_size=config["input_size"],
        batch_size=config["batch_size"],
        num_workers=config["num_workers"],
        mean=config["mean"],
        std=config["std"],
    )

    print(
        f"Train batches: {len(backbone_train_loader)} | Val batches: {len(backbone_val_loader)} | Test batches: {len(test_loader)}"
    )

    return {
        "label_encoder": label_encoder,
        "num_classes": num_classes,
        "backbone_train_loader": backbone_train_loader,
        "backbone_val_loader": backbone_val_loader,
        "pairs_df": pairs_df,
        "test_loader": test_loader,
    }


## Submission Helper


In [4]:
def create_submission(
    backbone,
    head_model,
    device,
    pairs_df,
    test_loader,
    output_path,
):
    names, embeddings = inference.extract_embeddings_with_names_backbone(
        backbone,
        head_model,
        test_loader,
        device,
    )
    embedding_lookup = inference.build_embedding_lookup(names, embeddings)

    similarities = inference.compute_similarity_for_pairs(pairs_df, embedding_lookup)

    submission_df = pd.DataFrame({
        "row_id": pairs_df["row_id"],
        "similarity": similarities,
    })

    submission_df.to_csv(output_path, index=False)
    return submission_df


## Run Experiments


In [5]:
experiments = [
    # {"name": "MegaDescriptor", "backbone_name": "hf-hub:BVRA/MegaDescriptor-L-384", "input_size": 384},
    # {"name": "ResNet50", "backbone_name": "resnet50.a1_in1k", "input_size": 288},
    # {"name": "EfficientNetB3", "backbone_name": "efficientnet_b3", "input_size": 300},
    # {"name": "DinoV3", "backbone_name": "vit_large_patch16_dinov3.lvd1689m", "input_size": 256},
    {"name": "EVA02_Large", "backbone_name": "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k", "input_size": 448},
]


def run_experiment(experiment, run_idx, total_runs):
    backbone_name = experiment["backbone_name"]
    input_size = experiment["input_size"]

    print("=" * 80)
    print(f"Run {run_idx}/{total_runs}: {backbone_name} | input_size={input_size}")

    config["backbone_name"] = backbone_name
    config["input_size"] = input_size

    backbone = build_backbone(backbone_name, pretrained=True).to(device)
    backbone.eval()
    backbone_out_dim = getattr(backbone, "num_features", None)
    if backbone_out_dim is None:
        raise ValueError("Backbone output dimension not found; pass backbone_out_dim")

    data_bundle = load_data(config, backbone)
    label_encoder = data_bundle["label_encoder"]
    num_classes = data_bundle["num_classes"]
    backbone_train_loader = data_bundle["backbone_train_loader"]
    backbone_val_loader = data_bundle["backbone_val_loader"]
    pairs_df = data_bundle["pairs_df"]
    test_loader = data_bundle["test_loader"]

    cache_dir = config["checkpoint_dir"] / "embedding_cache"
    cache_dir.mkdir(exist_ok=True)
    cache_key = f"{backbone_name.replace(':', '_').replace('/', '_')}_{input_size}"

    train_cache = cache_dir / f"train_{cache_key}.npz"
    val_cache = cache_dir / f"val_{cache_key}.npz"

    train_embeddings, train_labels = feature_cache.get_or_create_embeddings(
        train_cache,
        backbone,
        backbone_train_loader,
        device,
    )
    val_embeddings, val_labels = feature_cache.get_or_create_embeddings(
        val_cache,
        backbone,
        backbone_val_loader,
        device,
    )

    train_loader, val_loader = data.create_embedding_dataloaders(
        train_embeddings,
        train_labels,
        val_embeddings,
        val_labels,
        batch_size=config["batch_size"],
    )

    head_model = ArcFaceHeadModel(
        input_dim=backbone_out_dim,
        num_classes=num_classes,
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        margin=config["arcface_margin"],
        scale=config["arcface_scale"],
        dropout=config["dropout"],
    ).to(device)

    param_count = sum(p.numel() for p in head_model.parameters())
    param_count_backbone = sum(p.numel() for p in backbone.parameters())

    run_name = f"{experiment['name']}"
    wandb = init_wandb(config, run_name=run_name, param_count=param_count, param_count_backbone=param_count_backbone)

    criterion = torch.nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        [p for p in head_model.parameters() if p.requires_grad],
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
    )

    checkpoint_name = f"{run_name}.pth"
    results = fit_embeddings(
        model=head_model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=config,
        device=device,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        label_encoder=label_encoder,
        wandb_run=wandb,
        checkpoint_name=checkpoint_name,
    )

    wandb.run.summary["best_val_mAP"] = results["best_map"]
    wandb.run.summary["best_val_loss"] = results["best_val_loss"]
    wandb.run.summary["best_epoch"] = results["best_epoch"]
    wandb.run.summary["total_epochs"] = results["epochs_ran"]

    checkpoint_path = config["checkpoint_dir"] / checkpoint_name
    submission_path = config["checkpoint_dir"] / f"submission_{run_name}.csv"
    submission_df = create_submission(
        backbone,
        head_model,
        device,
        pairs_df,
        test_loader,
        submission_path,
    )

    model_artifact = wandb.Artifact(
        name=run_name,
        type="model",
        description="checkpoint",
    )
    model_artifact.add_file(str(checkpoint_path))
    wandb.log_artifact(model_artifact)

    submission_artifact = wandb.Artifact(
        name=run_name,
        type="submission",
        description="Competition submission file",
    )
    submission_artifact.add_file(str(submission_path))
    wandb.log_artifact(submission_artifact)

    wandb.finish()
    print(f"Finished run {run_idx}")


for run_idx, experiment in enumerate(experiments, start=1):
    run_experiment(experiment, run_idx, len(experiments))


Run 1/1: eva02_large_patch14_448.mim_m38m_ft_in22k_in1k | input_size=448


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /sc/home/matthias.cram/.netrc


Train batches: 48 | Val batches: 12 | Test batches: 12


wandb: Currently logged in as: matthiascr (juggling-jaguars) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting training for 100 epochs...


...


Training complete!
Best epoch: 74 (Val Loss: 2.8392, Val mAP: 0.8628)


Embedding:   0%|          | 0/12 [00:00<?, ?it/s]

Computing similarities:   0%|          | 0/137270 [00:00<?, ?it/s]

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇███
learning_rate,█████████████████████████████████████▁▁▁
train_acc,▁▁▁▂▃▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█▇█████████████
train_loss,█▇▆▅▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▁▂▆▆▇▇▇▇▇▇█████████████████████████████
val_loss,█▅▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_map,▁▂▂▃▃▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇████████████████████
best_epoch,74
best_val_loss,2.83924
best_val_mAP,0.8628
epoch,84


Finished run 1
